# Transfer Learning & Fine-Tuning Strategies
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> Pretrained models, Feature Extraction vs Fine-Tuning, Discriminative Learning Rates, Catastrophic Forgetting, and Domain Adaptation.

---

### 📖 Conceptual Foundations

#### What is Transfer Learning?

**Transfer Learning** is the paradigm where knowledge acquired by training a model on a source domain $\mathcal{D}_S$ and task $\mathcal{T}_S$ (e.g. 1.2M ImageNet images or 100B tokens of web text) is transferred to improve learning on a target domain $\mathcal{D}_T$ and target task $\mathcal{T}_T$, where target labeled data is scarce or expensive to acquire.


#### Transfer Learning Strategy Matrix

| Target Dataset Size | Domain Similarity to Source | Recommended Fine-Tuning Strategy | Key Rationale |
| --- | --- | --- | --- |
| Small (< 1,000 samples) | High Similarity | **Frozen Backbone (Feature Extractor)**: Freeze all pre-trained layers; train only a new linear classification head. | Prevents overfitting on the small dataset; early and late layers already contain relevant filters. |
| Small (< 1,000 samples) | Low Similarity (e.g. Medical CT vs ImageNet) | **Train Early Layers + Linear Head**: Freeze deep task-specific layers, fine-tune early generic edge/texture layers. | High-level representations don't match, but low-level edge features transfer. |
| Large (> 50,000 samples) | High Similarity | **Full Model Fine-Tuning**: Initialize all weights from pretrained checkpoint; train entire network with small learning rate. | Abundant target data allows adapting deep representations without risk of catastrophic overfitting. |
| Large (> 50,000 samples) | Low Similarity | **Full Retraining from Pretrained Initialization**: Fine-tune all layers from checkpoint (or scratch). | Pretrained weights provide better loss basin initialization than random weights, converging significantly faster. |


#### The Threat of Catastrophic Forgetting

**Catastrophic Forgetting** occurs when fine-tuning a pretrained model on a new task causes it to abruptly overwrite and lose the general representations learned during pretraining.

**Proven Mitigations:**
1. **Discriminative Learning Rates (ULMFiT):** Use lower learning rates on early layers (e.g. $10^{-6}$) and higher learning rates on upper layers (e.g. $10^{-4}$).
2. **Gradual Unfreezing:** Train the output head first, then unfreeze the top layer, then progressively unfreeze deeper layers.
3. **Low Learning Rates with Warmup:** Scale base learning rate down by 10x to 100x compared to pretraining.


#### Domain Adaptation & Distribution Shift

When transferring between domains where input distributions differ ($P(X_S) \ne P(X_T)$) but the task is identical (e.g., synthetic gaming images to real-world self-driving car cameras), **Unsupervised Domain Adaptation (UDA)** techniques align representations:
- **Domain Adversarial Neural Networks (DANN):** Train a gradient reversal layer that penalizes the feature extractor if a domain classifier can distinguish source from target features.
- **Maximum Mean Discrepancy (MMD):** Minimizes distance between source and target feature distributions in a Reproducing Kernel Hilbert Space (RKHS).



In [ ]:
# %matplotlib inline
import numpy as np

# Conceptual representation of layer freezing and discriminative learning rates
class Layer:
    def __init__(self, name, size, requires_grad=True):
        self.name = name
        self.W = np.random.randn(*size) * 0.05
        self.requires_grad = requires_grad

# 4-layer model backbone + head
model = [
    Layer('conv1_stem', (16, 3, 3, 3), requires_grad=False), # Frozen
    Layer('conv2_mid',  (32, 16, 3, 3), requires_grad=False), # Frozen
    Layer('conv3_deep', (64, 32, 3, 3), requires_grad=True),  # Fine-tune
    Layer('linear_head', (10, 64), requires_grad=True)       # New Head
]

# Configure discriminative learning rates
learning_rates = {
    'conv3_deep': 1e-5,    # Gentle fine-tuning
    'linear_head': 1e-3    # Aggressive head adaptation
}

print('=== Transfer Learning Configuration ===')
for layer in model:
    status = 'TRAINABLE' if layer.requires_grad else 'FROZEN'
    lr = learning_rates.get(layer.name, 0.0)
    print(f'Layer: {layer.name:<12} | Status: {status:<9} | Learning Rate: {lr}')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

